# hexbot · Guided tour

One notebook, top to bottom. Takes around 30 minutes (less if you skip the training step, longer if you sit with the exercises). No prior framework experience required.

**For Colab users:** Runtime → Change runtime type → GPU before starting (only the training section actually needs a GPU).

**Table of contents**

1. Install + sanity check
2. Game basics: stones, turns, win detection
3. Reading the board
4. Analysis tools: threats, alpha-beta, solver
5. Writing your first bot in three steps
6. Plug your bot into the framework
7. Watch Orca think (MCTS)
8. Train Orca
9. Inspect the run (TensorBoard + manifest)
10. Compare your bot to the bundled one
11. Share via Model Zoo
12. Where to go next

After this notebook, the [advanced internals notebook](advanced_internals.ipynb) covers MCTS math, the policy/value heads, training loss, and the optimisations that let the loop run on a laptop.

## 1. Install + sanity check

In [ ]:
!pip install --quiet hexbot tensorboard

In [ ]:
from hexbot import HexGame, Bot, Arena

# One game between the bundled Orca and the rule-based heuristic.
result = Arena(Bot.orca(), Bot.heuristic(), num_games=1).play()
print(result)

## 2. Game basics: stones, turns, win detection

Hex Connect-6 plays on an **infinite hexagonal grid** with axial coordinates `(q, r)`. Player 0 places 1 stone on the very first turn, then both players alternate placing 2 stones per turn forever after (the 1-2-2 turn structure). Win condition: six of your own stones in an unbroken line along one of the three hex axes.

In [ ]:
game = HexGame()

# Turn 1: P0 places one stone
game.place(0, 0)
print(f"after P0's opening:    current={game.current_player}")

# Turn 2: P1 places two stones
game.place(2, 0)
print(f"after P1's first:      current={game.current_player}  (still P1)")
game.place(2, -1)
print(f"after P1's second:     current={game.current_player}  (back to P0)")

print(f"\ntotal stones placed: {game.total_stones}")

In [ ]:
# Scripted P0 win along the q axis
g = HexGame()
g.place(0, 0)
g.place(0, 5); g.place(0, 6)
g.place(1, 0); g.place(2, 0)
g.place(1, 5); g.place(1, 6)
g.place(3, 0); g.place(4, 0)
g.place(2, 5); g.place(2, 6)
g.place(5, 0)

print(f"is_over: {g.is_over}")
print(f"winner:  P{g.winner}")

**Try it yourself.** Build a P1 win along the `(1, -1)` diagonal (stones at `(k, -k)` for `k = 0..5`). Remember the 1-2-2 structure when sequencing P0's blocking attempts.

In [ ]:
# Your code here
g = HexGame()
# ...
if g.is_over:
    print(f"P{g.winner} wins")
else:
    print("game still in progress")

## 3. Reading the board

`legal_moves()` lists all candidate cells. `scored_moves(n)` ranks the top n by the C engine's heuristic (line extension potential + blocking value + proximity).

In [ ]:
g = HexGame()
for q, r in [(0,0), (3,0), (3,-1), (1,0), (2,0)]:
    g.place(q, r)

print(f"legal moves: {len(g.legal_moves())} cells")
print("\ntop 5 by heuristic:")
for q, r, score in g.scored_moves(5):
    print(f"  ({q:>3},{r:>3})  score={score}")

## 4. Analysis tools: threats, alpha-beta, solver

A **threat** is a move that creates an immediate winning option (a five-in-a-row that could become six on the next turn). `find_threats` and `find_winning_moves` are cheap lookups; `find_forced_move` returns the single move you must play if your opponent threatens an immediate win.

In [ ]:
from hexbot import find_threats, find_winning_moves, find_forced_move

# Same five-in-a-row setup as above, just before P0 wins
g = HexGame()
g.place(0, 0)
g.place(0, 5); g.place(0, 6)
g.place(1, 0); g.place(2, 0)
g.place(1, 5); g.place(1, 6)
g.place(3, 0); g.place(4, 0)
g.place(2, 5); g.place(2, 6)

print(f"P0 winning moves: {find_winning_moves(g, player=0)}")
print(f"P1 winning moves: {find_winning_moves(g, player=1)}")
print(f"forced move:      {find_forced_move(g)}")

**Alpha-beta search.** `game.search(depth=N)` does an alpha-beta search with transposition tables, killer heuristics, and late move reduction (all in C). Depth is measured in half-moves.

In [ ]:
g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

result = g.search(depth=6)
print(f"best move:  {result['best_move']}")
print(f"evaluation: {result['value']:+.2f}  (positive = good for side to move)")
print(f"nodes:      {result['nodes']:,}")

## 5. Writing your first bot in three steps

Anything callable as `bot(game) -> (q, r)` is a bot. We will start with the simplest possible bot, then make it smarter twice.

In [ ]:
from hexbot import evaluate_moves

# Step A: pick the top heuristic move
def top_bot(game):
    return evaluate_moves(game, top_n=1)[0][0]

# Step B: answer forced moves first (this single change is the biggest jump)
def safer_bot(game):
    forced = find_forced_move(game)
    if forced is not None:
        return forced
    return evaluate_moves(game, top_n=1)[0][0]

# Step C: alpha-beta with the forced-move shortcut
def deep_bot(game, depth=6):
    forced = find_forced_move(game)
    if forced is not None:
        return forced
    return game.search(depth=depth)['best_move']

In [ ]:
# Pit them in Arena
print('deep_bot vs top_bot:        ', Arena(deep_bot, top_bot, num_games=6).play(verbose=False))
print('deep_bot vs Bot.heuristic():', Arena(deep_bot, Bot.heuristic(), num_games=4).play(verbose=False))
print('deep_bot vs Bot.random():   ', Arena(deep_bot, Bot.random(), num_games=4).play(verbose=False))

## 6. Plug your bot into the framework

Anything in the framework that accepts a bot name (`--opponent`, the dashboard dropdown, the leaderboard) finds your bot by string once you register it.

In [ ]:
from hexbot import BotProtocol, register_bot, registered_bots

class DeepBot(BotProtocol):
    def __init__(self, depth=6):
        self.depth = depth
    def best_move(self, game):
        return deep_bot(game, depth=self.depth)

register_bot('deep', DeepBot)
print('registered bots:', sorted(registered_bots().keys()))

## 7. Watch Orca think (MCTS)

Orca is an AlphaZero-style neural network: a policy head (probability distribution over moves) plus a value head (who is winning). At inference time it runs Monte Carlo Tree Search guided by both heads.

The output of `mcts_search` shows visit counts. A move with more visits is one MCTS considered more promising. The top-visited move is the one played.

In [ ]:
from hexbot import mcts_search

g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

result = mcts_search(g, sims=200)
print(f"best move: {result['best_move']}")
print()
for move, visits in result['top_moves'][:5]:
    bar = '#' * (visits // 4)
    print(f"  {move}  visits={visits:>3}  {bar}")

## 8. Train Orca

Twenty self-play iterations with the `colab-t4` hardware profile. Takes a few minutes on a T4 GPU. Loss / ELO / lr / iteration time are logged to TensorBoard under `runs/<timestamp>/`.

Even 20 iterations won't make a much stronger bot than the bundled one. The point of this step is to see the loop run and the metrics move.

In [ ]:
!python -m orca.train --profile=colab-t4 --iterations 20 --tensorboard

## 9. Inspect the run (TensorBoard + manifest)

TensorBoard inline. Watch `loss/total` come down and `elo/current` climb.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

In [ ]:
import json, glob

manifest_path = sorted(glob.glob('runs/*/manifest.json'))[-1]
manifest = json.load(open(manifest_path))
print(f"Manifest: {manifest_path}")
for key in ['run_id', 'hexbot_version', 'git_sha', 'hostname', 'device']:
    print(f"  {key:20s} {manifest.get(key, '?')}")
print('\n  config snapshot:')
for k, v in manifest['config'].items():
    print(f"    {k:24s} {v}")

## 10. Compare your bot to the bundled one

In [ ]:
import torch

ckpts = sorted(glob.glob('hex_checkpoint_*.pt'),
               key=lambda p: int(p.split('_')[-1].split('.')[0]))
latest = ckpts[-1]
meta = torch.load(latest, weights_only=False)['_hexbot_meta']
print(f"latest: {latest}  (iter={meta['iter']}, arch={meta['arch']})")

trained = Bot.from_checkpoint(latest)
bundled = Bot.orca()
print(Arena(trained, bundled, num_games=4).play(verbose=False))

## 11. Share via Model Zoo

Package a checkpoint with metadata so others can `Zoo.download('your-name')`.

In [ ]:
from orca.zoo import Zoo

Zoo.package(
    latest, output_path='my-first-orca.pt',
    name='my-first-orca', author='you',
    elo=int(meta.get('elo') or 1000),
    description=f"Trained {meta['iter']} iterations in the guided tour",
)
Zoo.list()

Publishing for real (`Zoo.upload`) needs the `gh` CLI installed and authenticated; once your bot lands on the leaderboard, the [Featured Community Bots](https://github.com/Saiki77/hexbot-building-framework#featured-community-bots) table auto-regenerates and you show up on the repo front page.

## 12. Where to go next

| Goal | Read |
|---|---|
| Understand MCTS, policy/value heads, training loss, and the optimisations under the hood | [Advanced internals notebook](advanced_internals.ipynb) |
| Go past 20 iterations | [Training Guide](https://github.com/Saiki77/hexbot-building-framework/wiki/Training-Guide) |
| Hyperparameter sweep with Optuna | `python -m orca.sweep --trials 20` (requires `pip install 'hexbot[sweep]'`) |
| Warm-start from human games | [SFT Guide](https://github.com/Saiki77/hexbot-building-framework/wiki/SFT-Guide) |
| Play your bot on hexo.did.science | [Playing Online](https://github.com/Saiki77/hexbot-building-framework/wiki/Playing-Online) |
| Full feature reference | [Wiki home](https://github.com/Saiki77/hexbot-building-framework/wiki) |